# 04 — OCR baseline evaluation and error analysis

This notebook reads local artifacts produced by `scripts/run_ocr.py` and `scripts/evaluate_ocr.py`. It compares full-page transcription annotations with Tesseract word boxes using the documented center-in-polygon alignment. Full-page annotation polygons have coarser granularity than word boxes, so detection F1 is reported as N/A. Receipt images and text remain local; clear outputs before committing.

Run the commands below first from the repository root:

```bash
uv run python scripts/run_ocr.py --split internal_dev --output-dir outputs/ocr/tesseract_internal_dev
uv run python scripts/evaluate_ocr.py --split internal_dev --output-dir outputs/ocr/evaluation
```


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from matplotlib.patches import Polygon
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from docucheck.evaluation.metrics import align_word_predictions_to_lines
from docucheck.ocr.inference import load_manifest

MANIFEST = load_manifest(ROOT / "data/processed/manifest.jsonl")
PREDICTION_ROWS = load_manifest(ROOT / "outputs/ocr/tesseract_internal_dev/predictions.jsonl")
METRICS = json.loads((ROOT / "outputs/ocr/evaluation/metrics.json").read_text(encoding="utf-8"))
PREDICTIONS = {row["image_id"]: row for row in PREDICTION_ROWS}
RECORDS = {row["image_id"]: row for row in MANIFEST if row.get("split") == "internal_dev"}
DATASET_ROOT = Path(os.environ.get("MC_OCR_ROOT", ROOT / "data/mcocr")).expanduser()
print(f"Images with transcription GT: {METRICS['images_with_full_page_ground_truth']}")
print(f"Images evaluated with OCR: {METRICS['images_with_ground_truth_and_predictions']}")
print(f"Evaluated GT regions: {METRICS['evaluated_ground_truth_regions']:,}; excluded invalid geometry: {METRICS['excluded_invalid_geometry_regions']:,}")
print(f"CER: {METRICS['cer']:.3f}; WER: {METRICS['wer']:.3f}; detection: {METRICS['detection_metrics']['status']}")
print(f"GT regions without assigned words: {METRICS['unmatched_ground_truth_regions']:,}")
print(f"Predicted word boxes outside valid GT: {METRICS['unassigned_prediction_word_boxes']:,}")
print("Alignment: assign each word by box center to the smallest valid GT polygon, then sort by y and x.")

## Aggregate quality and image-level error distribution

CER is micro-averaged over normalized characters, and WER uses whitespace-separated Vietnamese syllables. CER/WER can exceed 1 because insertions are counted; many predicted boxes fall outside annotated polygons and some reference regions receive no prediction. These results describe this alignment protocol and are not KIE field extraction scores.

In [ ]:
image_metrics = load_manifest(ROOT / "outputs/ocr/evaluation/per_image_metrics.jsonl")
df = pd.DataFrame(image_metrics)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for column, title, ax in [("cer", "Image-level CER", axes[0]), ("wer", "Image-level WER", axes[1])]:
    values = pd.to_numeric(df[column], errors="coerce").dropna()
    ax.hist(values.clip(upper=5), bins=25, color="#287271", edgecolor="white")
    ax.axvline(values.median(), color="#e76f51", linestyle="--", label=f"median={values.median():.2f}")
    ax.set(title=title + " (values >5 clipped for display)", xlabel="Error rate", ylabel="Images")
    ax.legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df["gt_characters"], df["cer"].clip(upper=5), alpha=.55, s=22, color="#e76f51")
ax.set(xlabel="Ground-truth characters per image", ylabel="CER (display clipped at 5)", title="Error versus transcript length")
plt.tight_layout(); plt.show()
print("Image-level groups with fewer than 10 examples are not interpreted separately.")

## Local annotated image gallery

Green polygons show word-level Tesseract predictions; magenta outlines show full-page GT regions. The GT geometry is coarser, and text labels are included only in this local interactive view. A high-error case and a lower-error case are selected from actual evaluation results.

In [ ]:
def show_case(image_id):
    record = RECORDS[image_id]
    path = DATASET_ROOT / record["image_path"]
    image = Image.open(path).convert("RGB")
    pred_row = PREDICTIONS.get(image_id, {})
    predictions = pred_row.get("predictions", []) if pred_row.get("status") == "ok" else []
    gt = [r for r in record.get("ocr_regions", []) if r.get("text_raw") and r.get("polygon") and not r.get("validation_issues")]
    aligned, _ = align_word_predictions_to_lines(predictions, gt)
    fig, ax = plt.subplots(figsize=(10, 12))
    ax.imshow(image)
    for p in predictions:
        ax.add_patch(Polygon(p["polygon"], fill=False, edgecolor="#16a085", linewidth=.8, alpha=.8))
    for region in gt:
        ax.add_patch(Polygon(region["polygon"], fill=False, edgecolor="#d63384", linewidth=1.3, alpha=.85))
    ax.plot([], [], color="#16a085", label="Tesseract word box")
    ax.plot([], [], color="#d63384", label="Full-page GT region")
    ax.legend(loc="upper right")
    ax.set_title(f"Local image {image_id}: {len(predictions)} predictions, {len(gt)} valid GT regions")
    ax.axis("off"); plt.tight_layout(); plt.show()
    comparison = pd.DataFrame([{"GT region": row["text_gt"], "Tesseract aligned text": row["text_pred"], "matched boxes": row["prediction_count"]} for row in aligned])
    display(comparison.head(12))

ordered = sorted((r for r in image_metrics if r.get("cer") is not None), key=lambda r: r["cer"])
chosen = [ordered[len(ordered)//2]["image_id"], ordered[-1]["image_id"]] if ordered else []
for image_id in chosen:
    show_case(image_id)

## Recognition-crop baseline gallery

This view uses the PSM 7 `official_val` crop baseline. Crop CER compares the emitted Tesseract words directly with each transcript, so polygon alignment does not affect the score. The gallery shows the highest-CER crops for local inspection; source transcripts can still contain annotation errors.

In [ ]:
crop_manifest = load_manifest(ROOT / "data/processed/recognition_manifest.jsonl")
crop_details = load_manifest(ROOT / "outputs/ocr/crop_psm7_official_val/per_crop_metrics.jsonl")
crop_records = {row["image_id"]: row for row in crop_manifest if row.get("split") == "official_val"}
assert crop_details and all(row["image_id"] in crop_records for row in crop_details)

crop_df = pd.DataFrame(crop_details)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(crop_df["cer"].clip(upper=1), bins=25, color="#287271", edgecolor="white")
ax.axvline(crop_df["cer"].median(), color="#e76f51", linestyle="--", label=f"median={crop_df['cer'].median():.2f}")
ax.set(title="PSM 7 crop CER (values >1 clipped for display)", xlabel="CER", ylabel="Crops")
ax.legend(); plt.tight_layout(); plt.show()

worst_crops = sorted(crop_details, key=lambda row: row["cer"] if row["cer"] is not None else -1, reverse=True)[:12]
fig, axes = plt.subplots(4, 3, figsize=(15, 10))
for ax, result in zip(axes.flat, worst_crops):
    image = Image.open(DATASET_ROOT / crop_records[result["image_id"]]["image_path"]).convert("RGB")
    ax.imshow(image)
    ax.set_title(f"CER={result['cer']:.2f}", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

display(pd.DataFrame([{"image_id": row["image_id"], "CER": row["cer"], "Ground truth": row["text_gt"], "Tesseract": row["text_pred"]} for row in worst_crops]))

## Three-engine `official_val` comparison

This table compares Tesseract PSM 7, VietOCR, and PaddleOCR on the same 1,300 recognition crops. Rows are ranked by micro CER. Latency is retained as run provenance, but it is not a speed ranking unless the engines run under controlled, equivalent CPU conditions. The galleries keep crop images and transcripts local.

In [ ]:
comparison = json.loads((ROOT / "outputs/ocr/engine_comparison/comparison.json").read_text(encoding="utf-8"))
per_crop_comparison = load_manifest(ROOT / "outputs/ocr/engine_comparison/per_crop_comparison.jsonl")
engine_keys = [row["engine_key"] for row in comparison["engines"]]
assert comparison["split"] == "official_val" and len(per_crop_comparison) == 1300

display(pd.DataFrame([{
    "Engine": row["engine"],
    "Model": row["model_names"]["recognition"],
    "CER": row["cer"],
    "WER": row["wer"],
    "Empty": row["empty_hypothesis_crops"],
    "Failures": row["failed_crops"],
    "Mean seconds/crop": row["mean_latency_seconds"],
} for row in comparison["engines"]]))

def error_summary(row):
    values = [row["engines"][key]["cer"] for key in engine_keys]
    return max(values), max(values) - min(values)

high_error = sorted(per_crop_comparison, key=lambda row: error_summary(row)[0], reverse=True)[:6]
disagreements = sorted((row for row in per_crop_comparison if row["disagreement"]), key=lambda row: error_summary(row)[1], reverse=True)[:6]
gallery_rows = list({row["image_id"]: row for row in high_error + disagreements}.values())
fig, axes = plt.subplots(4, 3, figsize=(15, 10))
for ax in axes.flat:
    ax.axis("off")
for ax, row in zip(axes.flat, gallery_rows):
    ax.imshow(Image.open(DATASET_ROOT / crop_records[row["image_id"]]["image_path"]).convert("RGB"))
    rates = " | ".join(f"{key}={row['engines'][key]['cer']:.2f}" for key in engine_keys)
    ax.set_title(rates, fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

display(pd.DataFrame([{
    "image_id": row["image_id"],
    "Ground truth": row["text_gt"],
    **{key: row["engines"][key]["text_pred"] for key in engine_keys},
} for row in gallery_rows]))

## Failure analysis

These counts come from the evaluation artifacts printed above. They describe the alignment between Tesseract words and full-page transcription annotations; inspect the image and transcript before assigning a cause.

1. **GT regions with no assigned words:** A valid annotation region received no Tesseract word-box center. OCR may have missed text, or the polygon/alignment may be unsuitable. Keep the no-evidence state explicit.
2. **Unassigned predicted word boxes:** Their centers fall outside every valid GT polygon. They may be extra OCR detections, text outside annotated coverage, or boxes excluded by invalid GT geometry. Do not count them all as false positives.
3. **Different annotation units:** GT polygons span lines or groups while Tesseract emits word boxes. A one-to-one IoU score would mix segmentation differences with detection quality. Define compatible annotation units before comparing detection systems.

Detection precision/recall/F1 remains **N/A** for this pairing. CER/WER are diagnostic for this alignment protocol, not KIE field extraction scores.
